In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "data").exists():
    raise FileNotFoundError("Open this notebook from the NutriCore repository or its notebooks directory.")

NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))
os.chdir(PROJECT_ROOT)

# Nutrition Target Calculation Module

## Project Overview

The goal of this notebook is to develop the nutrition calculation component of the NutriCore recommendation workflow.

Before recipes can be recommended, the system must first determine the user's nutritional requirements based on their personal characteristics and fitness goals.

The nutrition module will:

- Calculate Basal Metabolic Rate (BMR)
- Calculate Total Daily Energy Expenditure (TDEE)
- Generate calorie targets for different goals
- Calculate macronutrient requirements
- Support both automatic and manual nutrition planning

The outputs generated in this notebook will later be used by the recommendation engine to identify recipes that best match a user's nutritional needs.

This notebook focuses solely on nutrition calculations and does not perform recipe recommendation.

In [2]:
# Imports

import pandas as pd

pd.set_option('display.max_rows', None, 'display.max_columns', None,"display.max_colwidth", None)

## Basal Metabolic Rate (BMR)

Basal Metabolic Rate (BMR) represents the number of calories the body requires to perform essential physiological functions while at rest.

Examples include:

- Breathing
- Blood circulation
- Body temperature regulation
- Cell maintenance

This project uses the Mifflin-St Jeor Equation, one of the most widely accepted equations for estimating BMR.

<h3>Male</h3>

BMR = (10 × Weight) + (6.25 × Height) − (5 × Age) + 5

<h3>Female</h3>

BMR = (10 × Weight) + (6.25 × Height) − (5 × Age) − 161

Where:

- Weight is measured in kilograms (kg)
- Height is measured in centimeters (cm)
- Age is measured in years

In [3]:
# BMR Calculation Function

def calculate_bmr(age, gender, weight, height):

    gender = gender.lower()

    if gender == "male":
        return (10 * weight) + (6.25 * height) - (5 * age) + 5

    elif gender == "female":
        return (10 * weight) + (6.25 * height) - (5 * age) - 161

    else:
        raise ValueError(
            "Gender must be either 'male' or 'female'"
        )

### Testing BMR Calculation

Before integrating the function into the recommendation system, it is important to verify that the implementation produces realistic outputs.

A sample user profile will be used for validation.

In [4]:
# Example User

sample_bmr = calculate_bmr(
    age=25,
    gender="male",
    weight=75,
    height=175
)

print(f"BMR: {sample_bmr:.2f} kcal/day")

BMR: 1723.75 kcal/day


## Total Daily Energy Expenditure (TDEE)

While BMR represents the calories required at complete rest, most individuals perform physical activities throughout the day.

To estimate daily calorie expenditure more accurately, an activity multiplier is applied to the BMR.

This value is known as Total Daily Energy Expenditure (TDEE).

TDEE estimates the total number of calories required to maintain current body weight.

The following activity multipliers are used:

| Activity Level | Multiplier |
|---------------|------------|
| Sedentary | 1.20 |
| Lightly Active | 1.375 |
| Moderately Active | 1.55 |
| Very Active | 1.725 |
| Athlete | 1.90 |

The calculated TDEE serves as the foundation for generating calorie targets for maintenance, fat loss, and muscle gain.

In [5]:
# Activity Multipliers

ACTIVITY_MULTIPLIERS = {
    "sedentary": 1.20,
    "lightly active": 1.375,
    "moderately active": 1.55,
    "very active": 1.725,
    "athlete": 1.90
}

In [6]:
# TDEE Calculation Function

def calculate_tdee(bmr, activity_level):

    activity_level = activity_level.lower()

    if activity_level not in ACTIVITY_MULTIPLIERS:
        raise ValueError(
            "Invalid activity level provided."
        )

    return (
        bmr *
        ACTIVITY_MULTIPLIERS[activity_level]
    )

### Testing TDEE Calculation

A sample user profile is used to verify that the calculated maintenance calories are realistic.

The result should be significantly higher than the user's BMR because it incorporates daily activity levels.

In [7]:
# Sample TDEE Calculation

sample_tdee = calculate_tdee(
    bmr=sample_bmr,
    activity_level="moderately active"
)

print(
    f"TDEE: {sample_tdee:.2f} kcal/day"
)

TDEE: 2671.81 kcal/day


## Generating Goal-Based Calorie Targets

Different fitness goals require different calorie intakes.

Rather than applying fixed calorie adjustments, this project uses percentage-based modifications relative to a user's Total Daily Energy Expenditure (TDEE).

A percentage-based approach scales appropriately across individuals with different maintenance calorie requirements and provides more personalized recommendations.

The following calorie targets are generated:

| Goal | Calorie Adjustment |
|--------|------------------|
| Maintenance | No Change (100% of TDEE) |
| Mild Cut | −10% Calories (90% of TDEE) |
| Moderate Cut | −20% Calories (80% of TDEE) |
| Aggressive Cut | −25% Calories (75% of TDEE) |
| Lean Bulk | +5% Calories (105% of TDEE) |
| Bulk | +10% Calories (110% of TDEE) |
| Aggressive Bulk | +15% Calories (115% of TDEE) |

These calorie targets serve as the foundation for subsequent macronutrient calculations and recipe recommendations.

In [8]:
# Goal Based Calorie Targets

def generate_calorie_targets(tdee):

    return {
        "Maintenance": round(tdee),

        "Mild Cut": round(tdee * 0.90),

        "Moderate Cut": round(tdee * 0.80),

        "Aggressive Cut": round(tdee * 0.75),

        "Lean Bulk": round(tdee * 1.05),

        "Bulk": round(tdee * 1.10),

        "Aggressive Bulk": round(tdee * 1.15)
    }

In [9]:
# Example Calorie Targets

calorie_targets = generate_calorie_targets(
    sample_tdee
)

pd.DataFrame(
    calorie_targets.items(),
    columns=["Goal", "Calories"]
)

,Goal,Calories
0,Maintenance,2672
1,Mild Cut,2405
2,Moderate Cut,2137
3,Aggressive Cut,2004
4,Lean Bulk,2805
5,Bulk,2939
6,Aggressive Bulk,3073


### Goal Target Findings

The calculated calorie targets demonstrate how daily energy intake changes according to different fitness objectives.

As expected:

- Cutting phases reduce calorie intake below maintenance levels.
- Bulking phases increase calorie intake above maintenance levels.
- More aggressive goals result in larger deviations from maintenance calories.

These calorie targets will serve as inputs for macronutrient calculations in the next stage of the nutrition pipeline.

## Macronutrient Target Calculation

After determining calorie requirements, the next step is to calculate macronutrient targets.

Macronutrients are nutrients required in large quantities and provide energy to the body.

The three primary macronutrients are:

- Protein
- Carbohydrates
- Fat

Protein requirements vary depending on the user's fitness goal, while fat intake is determined as a percentage of total calories. Remaining calories are allocated to carbohydrates.

This approach provides a practical and flexible framework for generating personalized nutrition plans.

### Protein Requirements

Protein plays a critical role in:

- Muscle growth
- Muscle preservation
- Recovery
- Satiety

Protein requirements differ depending on whether the user is attempting to lose fat, maintain weight, or gain muscle.

The following protein targets are used:

| Goal | Protein Target |
|--------|---------------|
| Maintenance | 1.6 g/kg |
| Mild Cut | 2.0 g/kg |
| Moderate Cut | 2.2 g/kg |
| Aggressive Cut | 2.4 g/kg |
| Lean Bulk | 2.0 g/kg |
| Bulk | 2.0 g/kg |
| Aggressive Bulk | 2.0 g/kg |

These values are expressed per kilogram of body weight and are commonly used in sports nutrition for active individuals.

In [10]:
# Protein Targets (g/kg Body Weight)

PROTEIN_TARGETS = {

    "Maintenance": 1.6,

    "Mild Cut": 2.0,

    "Moderate Cut": 2.2,

    "Aggressive Cut": 2.4,

    "Lean Bulk": 2.0,

    "Bulk": 2.0,

    "Aggressive Bulk": 2.0
}

In [11]:
# Protein Calculation

def calculate_protein(weight, goal):

    return round(
        weight *
        PROTEIN_TARGETS[goal],
        1
    )

### Fat Requirements

Dietary fat is essential for:

- Hormone production
- Cell function
- Vitamin absorption
- Long-term energy storage

Most nutritional guidelines recommend that approximately 20–35% of total calories come from dietary fat.

For this project, 25% of total calories are allocated to fat intake. This provides a balanced approach that supports overall health while leaving sufficient calories available for protein and carbohydrates.

Since fat provides 9 calories per gram:

Fat (g) = Fat Calories ÷ 9

In [12]:
# Fat Calculation

def calculate_fat(calories):

    fat_calories = calories * 0.25

    return round(
        fat_calories / 9,
        1
    )

### Carbohydrate Requirements

After allocating calories to protein and fat, the remaining calories are assigned to carbohydrates.

Carbohydrates serve as the body's primary energy source and play an important role in:

- Physical performance
- Recovery
- Daily activities
- Exercise training

Since carbohydrates provide 4 calories per gram:

Carbohydrates (g) = Remaining Calories ÷ 4

In [13]:
# Carbohydrate Calculation

def calculate_carbs(
    calories,
    protein_grams,
    fat_grams
):

    protein_calories = protein_grams * 4

    fat_calories = fat_grams * 9

    remaining_calories = (
        calories
        - protein_calories
        - fat_calories
    )

    return round(
        remaining_calories / 4,
        1
    )

### Complete Macronutrient Profile Generation

The following function combines all previous calculations and generates a complete nutrition profile consisting of:

- Calories
- Protein
- Fat
- Carbohydrates

The generated nutrition profile will later be used by the recipe recommendation engine to identify recipes that best match a user's dietary requirements.

In [14]:
# Macro Target Generator

def generate_macro_targets(
    weight,
    calories,
    goal
):

    protein = calculate_protein(
        weight,
        goal
    )

    fat = calculate_fat(
        calories
    )

    carbs = calculate_carbs(
        calories,
        protein,
        fat
    )

    return {
        "Calories": round(calories),
        "Protein_g": protein,
        "Fat_g": fat,
        "Carbs_g": carbs
    }

### Testing Macronutrient Calculations

A sample user profile is used to verify that the generated macronutrient targets are realistic and internally consistent.

The example below calculates macronutrient targets for a user weighing 75 kg following a Mild Cut nutrition plan.

In [15]:
# Example Macro Calculation

sample_macros = generate_macro_targets(
    weight=75,
    calories=2405,
    goal="Mild Cut"
)

sample_macros

{'Calories': 2405, 'Protein_g': 150.0, 'Fat_g': 66.8, 'Carbs_g': 301.0}

## Generating Nutrition Profiles for All Goals

Users may wish to compare multiple nutrition strategies before selecting a specific goal.

To facilitate this process, nutrition profiles are generated for every supported goal.

Each profile contains:

- Daily Calories
- Protein Target
- Fat Target
- Carbohydrate Target

This allows users to evaluate the nutritional implications of different cutting and bulking strategies.

In [16]:
# Nutrition Profiles For All Goals

def generate_all_goal_profiles(
    weight,
    calorie_targets
):

    profiles = []

    for goal, calories in calorie_targets.items():

        macros = generate_macro_targets(
            weight=weight,
            calories=calories,
            goal=goal
        )

        profiles.append({
            "Goal": goal,
            "Calories": macros["Calories"],
            "Protein_g": macros["Protein_g"],
            "Fat_g": macros["Fat_g"],
            "Carbs_g": macros["Carbs_g"]
        })

    return pd.DataFrame(profiles)

### Example Nutrition Profiles

The following example generates nutrition profiles for all supported goals using the previously calculated calorie targets.

In [17]:
# Generate All Nutrition Profiles

nutrition_profiles = generate_all_goal_profiles(
    weight=75,
    calorie_targets=calorie_targets
)

nutrition_profiles

,Goal,Calories,Protein_g,Fat_g,Carbs_g
0,Maintenance,2672,120.0,74.2,381.0
1,Mild Cut,2405,150.0,66.8,301.0
2,Moderate Cut,2137,165.0,59.4,235.6
3,Aggressive Cut,2004,180.0,55.7,195.7
4,Lean Bulk,2805,150.0,77.9,376.0
5,Bulk,2939,150.0,81.6,401.1
6,Aggressive Bulk,3073,150.0,85.4,426.1


## Manual Nutrition Targets

While automated nutrition calculations are suitable for most users, some individuals may prefer to define their own calorie and macronutrient targets.

Examples include:

- Athletes following coach-designed nutrition plans
- Users tracking specific macro goals
- Advanced users with customized dietary requirements
- Individuals transitioning from another nutrition application

To support these use cases, the system provides a manual nutrition mode.

In manual mode:

- Calories are required
- Protein is required
- Fat is optional
- Carbohydrates are optional

Any missing macronutrients are automatically calculated while maintaining consistency with the specified calorie target.

### Manual Nutrition Calculation Logic

The manual nutrition engine supports multiple input scenarios.

<h3>Scenario 1</h3>

User provides:

- Calories
- Protein

System calculates:

- Fat
- Carbohydrates

using default nutrition assumptions.

<h3>Scenario 2</h3>

User provides:

- Calories
- Protein
- Fat

System calculates:

- Carbohydrates

<h3>Scenario 3</h3>

User provides:

- Calories
- Protein
- Carbohydrates

System calculates:

- Fat

<h3>Scenario 4</h3>

User provides:

- Calories
- Protein
- Fat
- Carbohydrates

System validates whether the supplied macronutrients approximately match the specified calorie target.

In [18]:
def generate_manual_targets(
    calories,
    protein,
    fat=None,
    carbs=None
):

    if fat is None and carbs is None:

        fat = round(
            (calories * 0.25) / 9,
            1
        )

        remaining_calories = (
            calories
            - (protein * 4)
            - (fat * 9)
        )

        carbs = round(
            remaining_calories / 4,
            1
        )

    elif fat is not None and carbs is None:

        remaining_calories = (
            calories
            - (protein * 4)
            - (fat * 9)
        )

        carbs = round(
            remaining_calories / 4,
            1
        )

    elif fat is None and carbs is not None:

        remaining_calories = (
            calories
            - (protein * 4)
            - (carbs * 4)
        )

        fat = round(
            remaining_calories / 9,
            1
        )

    return pd.DataFrame({
        "Calories": [calories],
        "Protein_g": [protein],
        "Fat_g": [fat],
        "Carbs_g": [carbs]
    })

### Nutrition Validation

When users manually enter macronutrient values, inconsistencies may arise between the specified calorie target and the calories implied by the selected macronutrients.

For example:

- Protein provides 4 kcal per gram
- Carbohydrates provide 4 kcal per gram
- Fat provides 9 kcal per gram

If the supplied macronutrients do not align with the specified calorie target, the generated nutrition profile may be inaccurate.

To address this issue, a validation step is performed by calculating the total calories contributed by all macronutrients and comparing this value against the user's target calories.

Small differences are expected due to rounding and are therefore considered acceptable.

In [19]:
# Macro Validation

def validate_macros(
    calories,
    protein,
    fat,
    carbs,
    tolerance=50
):

    macro_calories = (
        protein * 4 +
        fat * 9 +
        carbs * 4
    )

    difference = round(
        abs(calories - macro_calories),
        1
    )

    return pd.DataFrame({
        "Target Calories": [calories],
        "Macro Calories": [round(macro_calories, 1)],
        "Difference": [difference],
        "Valid": [difference <= tolerance]
    })

### Nutrition Quality Assessment

A calorie target may be mathematically valid while still producing an unusual macronutrient distribution.

For example:

- Extremely low protein intake may negatively affect recovery and muscle maintenance.
- Extremely low fat intake may affect hormone production and nutrient absorption.
- Extremely low carbohydrate intake may reduce training performance and energy availability.

To provide additional guidance, the system evaluates the percentage contribution of each macronutrient to total calorie intake.

The purpose of this assessment is not to restrict users but to identify potentially unusual nutritional patterns and provide advisory warnings where appropriate.

#### Quality Assessment Criteria

The system evaluates macronutrient distributions using broad nutritional guidelines.

| Macronutrient | Recommended Range |
|---------------|-------------------|
| Protein | 15% – 40% of Calories |
| Fat | 20% – 35% of Calories |
| Carbohydrates | Above 20% of Calories |

Values outside these ranges are flagged with advisory warnings.

These thresholds are intended as general guidance and do not represent medical recommendations.

In [20]:
def assess_macro_quality(
    calories,
    protein,
    fat,
    carbs
):

    warnings = []

    protein_pct = (
        protein * 4 / calories
    ) * 100

    fat_pct = (
        fat * 9 / calories
    ) * 100

    carb_pct = (
        carbs * 4 / calories
    ) * 100

    if protein_pct < 15:
        warnings.append(
            "Protein intake appears lower than recommended for active individuals."
        )

    elif protein_pct > 40:
        warnings.append(
            "Protein intake appears unusually high."
        )

    if fat_pct < 20:
        warnings.append(
            "Fat intake appears lower than recommended."
        )

    elif fat_pct > 35:
        warnings.append(
            "Fat intake appears higher than recommended."
        )

    if carb_pct < 20:
        warnings.append(
            "Carbohydrate intake appears lower than recommended."
        )

    assessment_df = pd.DataFrame({
        "Metric": [
            "Protein %",
            "Fat %",
            "Carbohydrates %"
        ],
        "Value": [
            round(protein_pct, 1),
            round(fat_pct, 1),
            round(carb_pct, 1)
        ]
    })

    warnings_df = pd.DataFrame({
        "Warnings": warnings if warnings else ["No Warnings"]
    })

    return assessment_df, warnings_df

### Recommendation Eligibility Check

Although unusual macro distributions may still be valid, they can reduce the effectiveness of recipe recommendations.

The recipe recommendation engine relies on matching user nutrition targets against recipes available in the Food.com dataset.

Extreme nutrition targets may not be adequately represented within the dataset and can therefore lead to poor recommendation quality.

To address this limitation, an eligibility assessment is performed.

The objective is not to prevent users from receiving recommendations, but rather to identify cases where recommendation quality may be reduced.

#### Eligibility Criteria

The eligibility check focuses on identifying extremely low macronutrient targets that are unlikely to be represented in typical recipe datasets.

Examples include:

- Protein targets approaching zero
- Extremely low fat intake
- Extremely low carbohydrate intake

When such cases are detected, the system generates a recommendation quality warning.

Users are still allowed to proceed, but they are informed that recommendation accuracy may be affected.

In [21]:
# Recommendation Eligibility

def check_recommendation_eligibility(
    calories,
    protein,
    fat,
    carbs
):

    eligible = True

    reasons = []

    protein_pct = (
        protein * 4 / calories
    ) * 100

    fat_pct = (
        fat * 9 / calories
    ) * 100

    carb_pct = (
        carbs * 4 / calories
    ) * 100

    if protein_pct < 10:

        eligible = False

        reasons.append(
            "Protein target is extremely low."
        )

    if fat_pct < 10:

        eligible = False

        reasons.append(
            "Fat target is extremely low."
        )

    if carb_pct < 5:

        eligible = False

        reasons.append(
            "Carbohydrate target is extremely low."
        )

    return pd.DataFrame({
        "Eligible": [eligible],
        "Reasons": [
            ", ".join(reasons)
            if reasons
            else "No Issues"
        ]
    })

## Example Scenarios

The following examples demonstrate how the manual nutrition engine behaves under both realistic and extreme nutritional configurations.

These examples illustrate:

- Macro calculation
- Nutrition validation
- Quality assessment
- Recommendation eligibility evaluation

### Example 1: Manual Nutrition Profile

The following example demonstrates a realistic nutrition profile.

The user specifies:

- Calories
- Protein
- Fat

The system automatically calculates the remaining carbohydrate requirement.

In [22]:
# Example 1

manual_profile = generate_manual_targets(
    calories=2200,
    protein=170,
    fat=60
)

manual_profile

,Calories,Protein_g,Fat_g,Carbs_g
0,2200,170,60,245.0


#### Validation Result

The generated nutrition profile is validated by comparing the target calorie intake against the calories implied by the supplied macronutrients.

In [23]:
validate_macros(
    calories=2200,
    protein=170,
    fat=60,
    carbs=245
)

,Target Calories,Macro Calories,Difference,Valid
0,2200,2200,0,True


In [24]:
assessment_df, warnings_df = assess_macro_quality(
    calories=2200,
    protein=170,
    fat=60,
    carbs=245
)

assessment_df

,Metric,Value
0,Protein %,30.9
1,Fat %,24.5
2,Carbohydrates %,44.5


In [25]:
warnings_df

,Warnings
0,No Warnings


#### Recommendation Eligibility

The following check determines whether the generated nutrition targets are likely to produce meaningful recipe recommendations.

This assessment focuses on recommendation quality rather than nutritional correctness.

In [26]:
check_recommendation_eligibility(
    calories=2200,
    protein=170,
    fat=60,
    carbs=245
)

,Eligible,Reasons
0,True,No Issues


### Example 2: Extreme Macro Distribution

The following example demonstrates how the system handles unusual nutrition targets.

Although mathematically valid, extreme macro distributions may reduce recommendation quality because similar recipes may not exist within the dataset.

The objective of this example is to demonstrate the warning and eligibility systems.

In [27]:
extreme_profile = generate_manual_targets(
    calories=2200,
    protein=10,
    carbs=20
)

extreme_profile

,Calories,Protein_g,Fat_g,Carbs_g
0,2200,10,231.1,20


In [28]:
extreme_fat = extreme_profile.loc[
    0,
    "Fat_g"
]

#### Extreme Profile Assessment

The generated profile is assessed using the same validation and recommendation quality procedures.

In [29]:
assessment_df, warnings_df = assess_macro_quality(
    calories=2200,
    protein=10,
    fat=extreme_fat,
    carbs=20
)

assessment_df

,Metric,Value
0,Protein %,1.8
1,Fat %,94.5
2,Carbohydrates %,3.6


In [30]:
warnings_df

,Warnings
0,Protein intake appears lower than recommended for active individuals.
1,Fat intake appears higher than recommended.
2,Carbohydrate intake appears lower than recommended.


In [31]:
check_recommendation_eligibility(
    calories=2200,
    protein=10,
    fat=extreme_fat,
    carbs=20
)

,Eligible,Reasons
0,False,"Protein target is extremely low., Carbohydrate target is extremely low."


### Discussion

The previous example demonstrates the distinction between:

1. Nutritional Validity
2. Recommendation Quality

A nutrition profile may be mathematically valid while still producing poor recipe recommendations due to limited representation within the dataset.

To address this issue, the system provides:

- Nutrition validation
- Quality assessment warnings
- Recommendation eligibility checks

These mechanisms improve transparency while preserving user flexibility.

## Conclusion

This notebook developed the nutrition calculation component of the NutriCore recommendation workflow.

The following functionalities were implemented:

- Basal Metabolic Rate (BMR) Calculation
- Total Daily Energy Expenditure (TDEE) Calculation
- Goal-Based Calorie Target Generation
- Macronutrient Target Calculation
- Nutrition Profile Generation for Multiple Fitness Goals
- Manual Nutrition Planning
- Macronutrient Validation
- Nutrition Quality Assessment
- Recommendation Eligibility Evaluation

The outputs generated in this notebook provide personalized calorie and macronutrient targets that will be used as inputs for the recipe recommendation engine.

The next stage of the project focuses on identifying recipes and serving sizes that best satisfy the nutritional requirements generated in this notebook.